# Module A2: Product Image Classifier Training
Download the real product checkout dataset using `kagglehub`, set up an image dataset generator for 5 retail classes (Clothing, Shoes, Bags, Electronics, Groceries), build a MobileNetV2 transfer learning model using TensorFlow/Keras, and serialize it.

In [1]:
import os
import json
import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 1. Dynamic Path Resolution: Auto-detects execution root
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == "notebooks":
    MODELS_DIR = os.path.join("..", "app", "models")
else:
    MODELS_DIR = os.path.join("app", "models")

os.makedirs(MODELS_DIR, exist_ok=True)
output_model_path = os.path.join(MODELS_DIR, "product_classifier.h5")

# 2. Download Fashion MNIST via kagglehub
print("Downloading 'zalando-research/fashionmnist' via kagglehub...")
dataset_path = kagglehub.dataset_download("zalando-research/fashionmnist")
print(f"Dataset stored at: {dataset_path}")

# Fashion MNIST contains 10 categories
CATEGORIES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", 
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

train_csv_path = os.path.join(dataset_path, "fashion-mnist_train.csv")

# 3. Process CSV Data into MobileNetV2 Tensor Shapes
if os.path.exists(train_csv_path):
    print("Loading and preprocessing Fashion MNIST dataset...")
    df = pd.read_csv(train_csv_path)
    
    # Replicate your exact limit: sample 40 records per class deterministically
    df_sampled = df.groupby('label').head(40).reset_index(drop=True)
    
    raw_labels = df_sampled['label'].values
    raw_pixels = df_sampled.drop(columns=['label']).values.astype(np.float32) / 255.0
    
    # Reshape the flattened 784 rows back into 28x28 single-channel matrices
    raw_pixels = raw_pixels.reshape(-1, 28, 28, 1)
    
    X_data = []
    print("Converting grayscale vectors into 224x224 RGB tensors...")
    for img in raw_pixels:
        # Convert 1-channel grayscale to 3-channel pseudo-RGB for MobileNetV2
        img_rgb = np.repeat(img, 3, axis=-1)
        
        # Resize from 28x28 up to 224x224 using native TensorFlow operations
        img_resized = tf.image.resize(img_rgb, (224, 224)).numpy()
        X_data.append(img_resized)
        
    X_data = np.array(X_data, dtype=np.float32)
    y_data = to_categorical(raw_labels, num_classes=10)
    
    print(f"[INFO] Prepared {len(X_data)} images across {len(CATEGORIES)} classes.")
else:
    raise FileNotFoundError(f"Could not locate fashion-mnist_train.csv at {train_csv_path}")

# 4. Transfer Learning Architecture Adjustments
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.2)(x)
# Adjusted output layers to map out 10 distinct Softmax probability nodes
predictions = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 5. Train and Save the Compiled Production Weights Package
print("Training product classifier on Fashion MNIST subset...")
model.fit(X_data, y_data, epochs=3, batch_size=16, verbose=1)

model.save(output_model_path)
print(f"[SUCCESS] Saved production model artifact to: {os.path.abspath(output_model_path)}")